# ISTAT IPAB — Indice dei Prezzi delle Abitazioni (v0)

**Fonte:** ISTAT SDMX, flow `143_497`, DATA_TYPE=59, MEASURE=4  
**Copertura:** 2010-Q1 → 2025-Q4, 8 aree geografiche, trimestrale  
**Aree:** Italia, 4 macro-aree (Nord-ovest, Nord-est, Centro, Mezzogiorno), 3 città (Milano, Roma, Torino)  
**Varianti:** NEW_DW (nuove costruzioni) · EXST_DW (abitazioni esistenti)

**Domanda:** Come sono cambiati i prezzi delle abitazioni in Italia dal 2010? Dove cresce di più? Qual è il divario tra nuovo ed esistente?

In [ ]:
import os
import duckdb
import pandas as pd

# Percorso mart — relativo alla root di dataset-incubator
MART = os.path.join(
    os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd(),
    'out', 'data', 'mart', 'istat_ipab_regionale', '2024', 'ipab_regionale.parquet'
)

db = duckdb.connect()
df = db.execute('SELECT * FROM read_parquet(?)', [MART]).fetchdf()

print(f'Righe totali:       {len(df):,}')
print(f'Colonne:            {list(df.columns)}')
print(f'Trimestri:          {df.trimestre.nunique()} ({df.trimestre.min()} → {df.trimestre.max()})')
print(f'Aree:               {df.area.nunique()} ({sorted(df.area.unique())})')
print(f'Valori nulli (indice_prezzi): {df.indice_prezzi.isna().sum()}')

## Sanity check

- **1.024 righe** = 8 aree × 2 tipi × 64 trimestri ✓  
- Serie completa 2010-Q1 → 2025-Q4, nessun valore nullo  
- Le aree coprono scala nazionale, macro-regionale e 3 città metropolitane

In [ ]:
# Distribuzione per livello geografico
df.groupby('livello_geografico')[['area']].nunique().rename(columns={'area': 'n_aree'})

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Trend nazionale: NEW_DW vs EXST_DW (base 2015=100)
it = df[df['area_codice'] == 'IT'].copy()
pivot = it.pivot(index='trimestre', columns='tipo_abitazione', values='indice_prezzi')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pivot.index, pivot['NEW_DW'], label='Nuove costruzioni (NEW_DW)', linewidth=2)
ax.plot(pivot.index, pivot['EXST_DW'], label='Abitazioni esistenti (EXST_DW)', linewidth=2, linestyle='--')
ax.axhline(100, color='gray', linewidth=0.8, linestyle=':')

# Evidenzia shock COVID e ripresa
ax.axvspan('2020-Q1', '2020-Q4', alpha=0.08, color='red', label='2020 (COVID)')

# Tick ogni anno (Q1)
q1_ticks = [t for t in pivot.index if t.endswith('Q1')]
ax.set_xticks(range(len(pivot.index)))
ticks_pos = [list(pivot.index).index(t) for t in q1_ticks]
ax.set_xticks(ticks_pos)
ax.set_xticklabels([t[:4] for t in q1_ticks], rotation=45)

ax.set_title('Italia — IPAB trimestrale (base 2015=100)', fontsize=13, pad=12)
ax.set_ylabel('Indice (base 2015=100)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nQ4-2025 vs Q1-2015:")
for t in ['NEW_DW', 'EXST_DW']:
    v_start = pivot.loc['2015-Q1', t]
    v_end = pivot.loc['2025-Q4', t]
    print(f"  {t}: {v_start:.1f} → {v_end:.1f}  ({(v_end/v_start - 1)*100:+.1f}%)")

In [ ]:
# Confronto macro-aree — ultimi 4 trimestri disponibili (2025)
macro = df[df['livello_geografico'] == 'macro_area'].copy()
last4 = macro[macro['trimestre'] >= '2025-Q1']

piv = last4.pivot_table(index='area', columns='tipo_abitazione', values='indice_prezzi', aggfunc='mean')
piv = piv.sort_values('NEW_DW', ascending=False)
print('Media 2025 per macro-area (base 2015=100):')
print(piv.round(1).to_string())

In [ ]:
# Confronto tutte le aree — variazione cumulata 2010-Q1 → 2025-Q4
start_q = '2010-Q1'
end_q = '2025-Q4'

s = df[df['trimestre'] == start_q].set_index(['area_codice', 'tipo_abitazione'])['indice_prezzi']
e = df[df['trimestre'] == end_q].set_index(['area_codice', 'tipo_abitazione'])['indice_prezzi']

var = ((e / s) - 1) * 100
var_df = var.reset_index().rename(columns={'indice_prezzi': 'var_pct'})
var_df = var_df.merge(df[['area_codice','area','livello_geografico']].drop_duplicates(), on='area_codice')
var_df = var_df.sort_values(['tipo_abitazione', 'var_pct'], ascending=[True, False])

print('Variazione cumulata 2010-Q1 → 2025-Q4 (%):')
print(var_df[['area', 'livello_geografico', 'tipo_abitazione', 'var_pct']].to_string(index=False))

In [ ]:
# Divario NEW_DW vs EXST_DW per area — tendenza recente
recent = df[df['trimestre'] >= '2020-Q1'].copy()
piv_all = recent.pivot_table(index=['area', 'livello_geografico', 'trimestre'],
                              columns='tipo_abitazione', values='indice_prezzi').reset_index()
piv_all['gap_new_minus_exst'] = piv_all['NEW_DW'] - piv_all['EXST_DW']

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
axes = axes.flatten()

aree = piv_all.groupby('area')['gap_new_minus_exst'].mean().sort_values(ascending=False).index

for i, area in enumerate(aree):
    sub = piv_all[piv_all['area'] == area].sort_values('trimestre')
    ax = axes[i]
    ax.fill_between(range(len(sub)), sub['gap_new_minus_exst'], 0, 
                    where=sub['gap_new_minus_exst'] > 0, alpha=0.4, color='steelblue', label='new > exst')
    ax.fill_between(range(len(sub)), sub['gap_new_minus_exst'], 0,
                    where=sub['gap_new_minus_exst'] <= 0, alpha=0.4, color='coral', label='new < exst')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(area, fontsize=10)
    ax.set_xticks([])
    ax.set_ylabel('NEW - EXST')

if len(aree) < len(axes):
    for j in range(len(aree), len(axes)):
        axes[j].set_visible(False)

fig.suptitle('Divario NEW_DW − EXST_DW per area (2020-Q1 → 2025-Q4)', fontsize=12)
plt.tight_layout()
plt.show()

## Lettura

**Tendenza generale:** dopo il rimbalzo post-COVID, i prezzi delle abitazioni in Italia sono in crescita continuata. Al **Q4-2025**, le nuove costruzioni crescono più delle abitazioni esistenti rispetto al 2010: il mercato del nuovo è più volatile e risponde prima alla domanda.

**Divario geografico:** il **Nord** (Nord-ovest e Nord-est) guida la crescita sia sul nuovo che sull'esistente. Il **Mezzogiorno** mostra dinamiche divergenti: forte crescita sul nuovo (potenziale segnale di domanda repressa o di campioni di mercato ridotti), stagnazione sull'esistente.

**Città:** **Milano** è il caso estremo — l'indice EXST_DW supera stabilmente il NEW_DW, segnalando un mercato dell'usato a premium. **Roma** e **Torino** mostrano dinamiche più moderate e più simili tra loro.

**Limiti:** la copertura geografica è limitata (4 macro-aree + 3 città metropolitane, nessuna regione NUTS2). Il dato non cattura la dispersione interna alle macro-aree, che è probabilmente sostanziale.

---

### Nota metodologica

| Voce | Dettaglio |
|---|---|
| Fonte | ISTAT SDMX, endpoint `esploradati.istat.it`, flow `IT1:143_497/1.0` |
| Indicatore | DATA_TYPE=59 (indice trimestrale IPAB), MEASURE=4 |
| Base | 2015=100 per la maggior parte delle serie; Roma usa base diversa |
| Frequenza | Trimestrale (FREQ=Q) |
| Copertura | 2010-Q1 → 2025-Q4 (64 trimestri) |
| Aree | IT, ITC, ITD, ITE, ITFG, ITC11 (Torino), ITC45 (Milano), ITE43 (Roma) |
| Pipeline | RAW (SDMX CSV) → CLEAN (DuckDB) → MART parquet via toolkit DataCivicLab |